# API & Product Documentation QA — How-To Answers With Citations

## 1. Project Overview

**Task:** Chat over API or product documentation, answer how-to questions, and cite the exact doc section that supports the answer.

**Approach:** Build realistic sample documentation → parse sections and code blocks → chunk with metadata → embed into ChromaDB → retrieve → generate step-by-step answers with citations → evaluate retrieval quality.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for answer generation
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store
- `LangChain` — text splitting

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Parse** documentation into sections, subsections, and code examples |
| 2 | **Chunk** docs so that code blocks and explanations stay together |
| 3 | **Retrieve** the right doc section for how-to questions |
| 4 | **Generate** step-by-step answers with inline section citations |
| 5 | **Handle** different question types: conceptual, how-to, troubleshooting, reference |
| 6 | **Evaluate** retrieval quality with ground-truth doc section mappings |

## 3. Problem Statement

Developers and product users constantly ask questions that are answered somewhere in the docs:

- *"How do I authenticate with the API?"*
- *"What rate limits apply to the search endpoint?"*
- *"How do I paginate through results?"*
- *"What error code do I get when the token expires?"*

They typically ctrl-F or skim headings. A RAG system can surface the right section instantly and present a concise answer with a link back to the source.

## 4. Why This Matters

- **Developer experience** — the #1 use case for doc chatbots in production
- **Code-aware retrieval** — code examples need special treatment (they embed poorly as prose)
- **Citation trust** — users must verify the answer against the official doc
- **Evaluation patterns** — production doc QA systems need retrieval accuracy metrics

## 5. Pipeline Architecture

```
Documentation Pages (markdown / restructuredtext / HTML)
       │
  ┌────┴──────────────────────────────────────┐
  │  Stage 1: DOC PARSING                      │
  │  • Split by headings (##, ###)             │
  │  • Detect code blocks (``` fenced)         │
  │  • Tag each chunk: page, section, has_code │
  └────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────┐
  │  Stage 2: CHUNKING                         │
  │  • Keep code block + surrounding text      │
  │  • Larger chunks (700 chars) for docs      │
  │  • Overlap so headings aren't lost         │
  └────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────┐
  │  Stage 3: EMBEDDING + INDEXING             │
  │  • all-MiniLM-L6-v2                       │
  │  • ChromaDB with section metadata          │
  └────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────┐
  │  Stage 4: QA GENERATION                    │
  │  • How-to → numbered steps + code          │
  │  • Reference → exact value quote           │
  │  • Troubleshooting → symptom/cause/fix     │
  │  • Inline [Doc: page § section] citations  │
  └────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────┐
  │  Stage 5: RETRIEVAL EVALUATION             │
  │  • Recall@k, MRR on ground-truth pairs    │
  │  • Section-level hit rate                  │
  └────────────────────────────────────────────┘
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface langchain-text-splitters chromadb sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

## 8. Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter
from typing import Optional

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 5
TEMP_ANSWER = 0.1
TEMP_JUDGE = 0.0

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Chunks: {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"Retrieval top-k: {TOP_K}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Sample Documentation

## 10. Build a Sample API Reference

We create a self-contained, realistic API doc for a fictional cloud storage service called **CloudVault**. It has multiple pages covering authentication, file operations, search, webhooks, rate limits, and error handling.

In [ ]:
DOCS = [
    {
        "page": "Authentication",
        "slug": "auth",
        "content": """## Authentication

All API requests require a bearer token in the `Authorization` header.

### Getting an API Key

1. Log in to the CloudVault dashboard at https://dashboard.cloudvault.io
2. Navigate to **Settings > API Keys**
3. Click **Generate New Key**
4. Copy the key immediately — it is shown only once

### Using the Token

Include the token in every request:

```bash
curl -H "Authorization: Bearer YOUR_API_KEY" \
     https://api.cloudvault.io/v2/files
```

### Token Expiry and Refresh

API keys do not expire by default. OAuth2 tokens expire after **3600 seconds** (1 hour).
Use the `/auth/refresh` endpoint to get a new access token:

```python
import requests

resp = requests.post(
    "https://api.cloudvault.io/v2/auth/refresh",
    json={"refresh_token": "your_refresh_token"}
)
new_token = resp.json()["access_token"]
```

### Scopes

Tokens can be scoped to limit access:

| Scope | Allows |
|-------|--------|
| `files:read` | List and download files |
| `files:write` | Upload, rename, and delete files |
| `search` | Use the search endpoint |
| `admin` | Manage users and settings |

Request scopes during key creation. A key with no scopes gets `files:read` by default."""
    },
    {
        "page": "File Operations",
        "slug": "files",
        "content": """## File Operations

### Uploading a File

Upload files using a multipart POST request:

```python
import requests

with open("report.pdf", "rb") as f:
    resp = requests.post(
        "https://api.cloudvault.io/v2/files/upload",
        headers={"Authorization": "Bearer YOUR_API_KEY"},
        files={"file": ("report.pdf", f, "application/pdf")},
        data={"folder_id": "folder_abc123"}
    )
print(resp.json())
# {"file_id": "file_xyz789", "size_bytes": 204800, "status": "uploaded"}
```

Maximum file size is **500 MB** for standard plans and **5 GB** for enterprise plans.

### Downloading a File

```python
resp = requests.get(
    "https://api.cloudvault.io/v2/files/file_xyz789/download",
    headers={"Authorization": "Bearer YOUR_API_KEY"},
    stream=True
)
with open("downloaded_report.pdf", "wb") as f:
    for chunk in resp.iter_content(chunk_size=8192):
        f.write(chunk)
```

### Listing Files in a Folder

```python
resp = requests.get(
    "https://api.cloudvault.io/v2/files",
    headers={"Authorization": "Bearer YOUR_API_KEY"},
    params={"folder_id": "folder_abc123", "page": 1, "per_page": 50}
)
files = resp.json()["files"]
print(f"Total: {resp.json()['total']}, Page: {resp.json()['page']}")
```

### Deleting a File

```python
resp = requests.delete(
    "https://api.cloudvault.io/v2/files/file_xyz789",
    headers={"Authorization": "Bearer YOUR_API_KEY"}
)
# Returns 204 No Content on success
```

Deleted files are moved to trash and permanently removed after **30 days**. Use `DELETE /files/{id}?permanent=true` to skip trash."""
    },
    {
        "page": "Search",
        "slug": "search",
        "content": """## Search

### Full-Text Search

Search across all files in your account by content or metadata:

```python
resp = requests.get(
    "https://api.cloudvault.io/v2/search",
    headers={"Authorization": "Bearer YOUR_API_KEY"},
    params={
        "q": "quarterly revenue report",
        "type": "pdf",
        "created_after": "2025-01-01"
    }
)
results = resp.json()["results"]
```

### Search Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `q` | string | Search query (required) |
| `type` | string | Filter by file type: `pdf`, `docx`, `xlsx`, `image` |
| `folder_id` | string | Limit search to a specific folder |
| `created_after` | ISO date | Only files created after this date |
| `created_before` | ISO date | Only files created before this date |
| `page` | int | Page number (default: 1) |
| `per_page` | int | Results per page (default: 20, max: 100) |

### Pagination

Search results are paginated. The response includes:

```json
{
  "results": [...],
  "total": 142,
  "page": 1,
  "per_page": 20,
  "next_page": "https://api.cloudvault.io/v2/search?q=...&page=2"
}
```

Use the `next_page` URL to fetch subsequent pages. When `next_page` is `null`, you have reached the last page."""
    },
    {
        "page": "Webhooks",
        "slug": "webhooks",
        "content": """## Webhooks

### Setting Up a Webhook

Webhooks notify your server when events occur in CloudVault.

```python
resp = requests.post(
    "https://api.cloudvault.io/v2/webhooks",
    headers={"Authorization": "Bearer YOUR_API_KEY"},
    json={
        "url": "https://yourapp.com/webhook",
        "events": ["file.uploaded", "file.deleted"],
        "secret": "your_webhook_secret"
    }
)
webhook_id = resp.json()["webhook_id"]
```

### Supported Events

| Event | Triggered When |
|-------|---------------|
| `file.uploaded` | A new file is uploaded |
| `file.deleted` | A file is deleted |
| `file.downloaded` | A file is downloaded |
| `folder.created` | A new folder is created |
| `user.invited` | A new user is invited to the workspace |

### Verifying Webhook Signatures

Every webhook request includes a `X-CloudVault-Signature` header. Verify it using HMAC-SHA256:

```python
import hmac
import hashlib

def verify_signature(payload: bytes, signature: str, secret: str) -> bool:
    expected = hmac.new(
        secret.encode(), payload, hashlib.sha256
    ).hexdigest()
    return hmac.compare_digest(f"sha256={expected}", signature)
```

### Retry Policy

CloudVault retries failed webhook deliveries up to **5 times** with exponential backoff (1s, 5s, 30s, 2min, 10min). After 5 failures the webhook is automatically disabled. Re-enable it from the dashboard or via `PATCH /webhooks/{id}`."""
    },
    {
        "page": "Rate Limits & Errors",
        "slug": "errors",
        "content": """## Rate Limits

### Default Limits

| Plan | Requests/min | Requests/day |
|------|-------------|-------------|
| Free | 60 | 10,000 |
| Standard | 300 | 100,000 |
| Enterprise | 1,000 | Unlimited |

### Rate Limit Headers

Every response includes rate limit headers:

- `X-RateLimit-Limit` — max requests per minute
- `X-RateLimit-Remaining` — remaining requests in the current window
- `X-RateLimit-Reset` — UTC timestamp when the window resets

When you exceed the limit, the API returns HTTP **429 Too Many Requests** with a `Retry-After` header (in seconds).

### Handling Rate Limits

```python
import time

def api_call_with_retry(url, headers, max_retries=3):
    for attempt in range(max_retries):
        resp = requests.get(url, headers=headers)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            print(f"Rate limited. Retrying in {wait}s...")
            time.sleep(wait)
            continue
        return resp
    raise Exception("Max retries exceeded")
```

## Error Codes

### Common Error Responses

| HTTP Code | Error | Meaning |
|-----------|-------|---------|
| 400 | `bad_request` | Invalid parameters or malformed JSON |
| 401 | `unauthorized` | Missing or invalid API key |
| 403 | `forbidden` | API key lacks required scope |
| 404 | `not_found` | Resource does not exist |
| 409 | `conflict` | File with same name already exists |
| 413 | `payload_too_large` | File exceeds size limit |
| 429 | `rate_limited` | Too many requests |
| 500 | `internal_error` | Server error — retry after 30s |

### Error Response Format

All errors return a consistent JSON body:

```json
{
  "error": {
    "code": "unauthorized",
    "message": "The provided API key is invalid or has been revoked.",
    "request_id": "req_abc123"
  }
}
```

Include the `request_id` when contacting support."""
    }
]

print(f"Documentation pages: {len(DOCS)}")
for doc in DOCS:
    code_blocks = len(re.findall(r"```", doc["content"])) // 2
    tables = doc["content"].count("| ")
    print(f"  [{doc['slug']:>10}] {doc['page']:25s} "
          f"{len(doc['content']):>5,} chars, {code_blocks} code blocks")

---

# Part B — Document Parsing & Chunking

## 11. Parse Documentation Sections

API documentation has a predictable structure: headings, prose, code blocks, tables. We parse each element and tag it with page + section metadata.

### Why Code Blocks Need Special Treatment

| Problem | Explanation |
|---------|-------------|
| **Code embeds poorly** | `requests.post("https://...")` is not natural language — embeddings struggle |
| **Splitting inside code** | A chunk boundary in the middle of a code block produces two broken snippets |
| **Context loss** | A code block without its heading and description is meaningless |

**Solution:** Keep each code block together with its surrounding text. We parse sections by heading and treat *heading + prose + code* as a single unit when possible.

In [ ]:
def parse_doc_sections(doc: dict) -> list[dict]:
    """Split a doc page into sections by ## and ### headings."""
    content = doc["content"]
    page = doc["page"]
    slug = doc["slug"]

    parts = re.split(r"(?=^#{2,3}\s)", content, flags=re.MULTILINE)
    sections = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        heading_match = re.match(r"^#{2,3}\s+(.+)", part)
        heading = heading_match.group(1).strip() if heading_match else page

        has_code = bool(re.search(r"```", part))
        has_table = bool(re.search(r"^\|.+\|", part, re.MULTILINE))

        sections.append({
            "text": part,
            "metadata": {
                "page": page,
                "slug": slug,
                "section": heading,
                "has_code": str(has_code).lower(),
                "has_table": str(has_table).lower(),
            },
        })
    return sections


all_sections = []
for doc in DOCS:
    all_sections.extend(parse_doc_sections(doc))

print(f"Total sections: {len(all_sections)}")
for s in all_sections:
    m = s["metadata"]
    code = "</>" if m["has_code"] == "true" else "   "
    table = "[T]" if m["has_table"] == "true" else "   "
    print(f"  {code} {table} [{m['slug']:>10}] {m['section'][:50]}")

## 12. Chunk the Sections

Sections that fit within `CHUNK_SIZE` stay intact. Longer sections are split with care not to break code blocks.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
)

all_chunks = []
for section in all_sections:
    text = section["text"]
    meta = section["metadata"]

    if len(text) <= CHUNK_SIZE:
        all_chunks.append({
            "text": text,
            "metadata": {**meta, "chunk_index": 0, "chunk_total": 1},
        })
    else:
        parts = splitter.split_text(text)
        for i, part in enumerate(parts):
            all_chunks.append({
                "text": part,
                "metadata": {**meta, "chunk_index": i, "chunk_total": len(parts)},
            })

print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk length: {sum(len(c['text']) for c in all_chunks) // len(all_chunks)} chars")

print("\nChunks per page:")
for page, count in Counter(c["metadata"]["page"] for c in all_chunks).items():
    print(f"  {page:25s}: {count}")

---

# Part C — Embedding, Indexing & Retrieval

## 13. Build the Vector Store

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("api_docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="api_docs",
    metadata={"hnsw:space": "cosine"},
)

texts = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]
vectors = embeddings.embed_documents(texts)
collection.add(documents=texts, embeddings=vectors, metadatas=metadatas, ids=ids)

print(f"Indexed {collection.count()} chunks in ChromaDB")

## 14. Retrieval Functions

In [ ]:
def retrieve(query: str, top_k: int = TOP_K,
             where: Optional[dict] = None) -> list[dict]:
    query_vector = embeddings.embed_query(query)
    kwargs = {"query_embeddings": [query_vector], "n_results": top_k}
    if where:
        kwargs["where"] = where
    results = collection.query(**kwargs)
    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return chunks


def display_chunks(chunks: list[dict], max_chars: int = 150):
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        sim = 1 - c["distance"]
        code = "</>" if m.get("has_code") == "true" else "   "
        print(f"\n  [{i}] {code} sim={sim:.3f} | {m['page']} > {m['section'][:40]}")
        print(f"      {textwrap.shorten(c['text'].replace(chr(10), ' '), width=max_chars, placeholder='...')}")


# Test
q = "How do I upload a file?"
print(f"Q: {q}")
chunks = retrieve(q, top_k=3)
display_chunks(chunks)

## 15. Filtered Retrieval

Metadata filters let you scope retrieval to a specific doc page, or prioritize chunks that contain code.

In [ ]:
# Filter: only auth page
q = "How do I refresh my token?"
print(f"Q: {q} (filter: auth page only)")
chunks = retrieve(q, where={"slug": "auth"})
display_chunks(chunks[:3])

# Filter: only chunks with code
q = "How do I handle rate limiting?"
print(f"\n{'='*60}")
print(f"Q: {q} (filter: has_code=true)")
chunks = retrieve(q, where={"has_code": "true"})
display_chunks(chunks[:3])

---

# Part D — Doc QA Answer Generation

## 16. Question Type Classification

Documentation questions fall into several categories. We classify the question first so the answer format matches the user's intent:

| Type | User Intent | Best Answer Format |
|------|------------|-------------------|
| **How-to** | "How do I upload a file?" | Numbered steps + code example |
| **Reference** | "What's the max file size?" | Direct value quote |
| **Troubleshooting** | "Why am I getting 401?" | Symptom → cause → fix |
| **Conceptual** | "What are scopes?" | Explanation + examples |
| **Listing** | "What events can I subscribe to?" | Bullet list or table |

In [ ]:
CLASSIFY_PROMPT = """Classify this documentation question into exactly one type.

QUESTION: {question}

Types: how_to, reference, troubleshooting, conceptual, listing

Return ONLY JSON: {{"type": "...", "reason": "one sentence"}}"""


def classify_question(question: str) -> str:
    resp = ask(CLASSIFY_PROMPT.format(question=question), temperature=0.0)
    parsed = parse_json(resp)
    if parsed and parsed.get("type"):
        return parsed["type"]
    return "how_to"


test_questions = [
    "How do I upload a file via the API?",
    "What is the maximum file size?",
    "Why am I getting a 403 error?",
    "What are webhook events?",
    "What scopes are available for API keys?",
]

for q in test_questions:
    qtype = classify_question(q)
    print(f"  [{qtype:>16}] {q}")

## 17. Doc QA System Prompt

## 18. Answer Generation Pipeline

In [ ]:
DOC_QA_SYSTEM = """You are a documentation assistant for the CloudVault API.
Answer questions using ONLY the provided doc sections.

Rules:
1. Cite every claim as [Doc: page > section].
2. For how-to questions, give numbered steps with code examples from the docs.
3. For reference questions, quote the exact value from the docs.
4. For troubleshooting, format as: Symptom > Cause > Fix.
5. If the docs don't cover the question, say so clearly.
6. Never invent endpoints, parameters, or behaviors not in the docs.
7. Include code blocks when relevant."""


def format_doc_sources(chunks: list[dict]) -> str:
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        parts.append(
            f"[Source {i}: {m['page']} > {m['section']}]\n"
            f"{c['text']}"
        )
    return "\n\n".join(parts)


def ask_docs(question: str,
             where: Optional[dict] = None) -> dict:
    """Full doc QA pipeline."""
    qtype = classify_question(question)
    chunks = retrieve(question, where=where)
    sources = format_doc_sources(chunks)

    prompt = (
        f"Answer this {qtype} question using ONLY the doc sections below.\n\n"
        f"QUESTION: {question}\n\n"
        f"DOC SECTIONS:\n{sources}\n\n"
        "Return ONLY JSON:\n"
        '{"answer": "detailed answer with [Doc: page > section] citations",'
        ' "question_type": "...",'
        ' "sources_used": ["page > section"],'
        ' "has_code_example": true,'
        ' "confidence": "high|medium|low",'
        ' "confidence_reason": "why"}'
    )

    resp = ask(prompt, system=DOC_QA_SYSTEM, temperature=TEMP_ANSWER)
    result = parse_json(resp)
    if not result:
        result = {
            "answer": resp,
            "question_type": qtype,
            "sources_used": [f"{c['metadata']['page']} > {c['metadata']['section']}" for c in chunks],
            "has_code_example": any(c["metadata"].get("has_code") == "true" for c in chunks),
            "confidence": "unknown",
            "confidence_reason": "LLM did not return valid JSON.",
        }
    result["chunks"] = chunks
    result["classified_type"] = qtype
    return result


def display_answer(question: str, result: dict):
    conf = result.get("confidence", "?")
    badge = {"high": "HIGH", "medium": "MED", "low": "LOW"}.get(conf, "???")
    qtype = result.get("classified_type", result.get("question_type", "?"))

    print(f"\n{'='*80}")
    print(f"Q: {question}")
    print(f"Type: {qtype} | Confidence: {badge}")
    print(f"{'='*80}\n")
    wrap_print(result.get("answer", ""))
    print(f"\n  Sources: {', '.join(result.get('sources_used', [])[:4])}")


print("Doc QA pipeline ready")

## 19. Ask How-To Questions

In [ ]:
q1 = "How do I upload a file using the API?"
r1 = ask_docs(q1)
display_answer(q1, r1)

In [ ]:
q2 = "How do I set up a webhook to be notified when files are uploaded?"
r2 = ask_docs(q2)
display_answer(q2, r2)

In [ ]:
q3 = "How do I authenticate my API requests?"
r3 = ask_docs(q3)
display_answer(q3, r3)

## 20. Ask Reference & Troubleshooting Questions

In [ ]:
q4 = "What is the maximum file size I can upload?"
r4 = ask_docs(q4)
display_answer(q4, r4)

In [ ]:
q5 = "What rate limits apply to my API key on the standard plan?"
r5 = ask_docs(q5)
display_answer(q5, r5)

In [ ]:
q6 = "I'm getting a 403 forbidden error. What's wrong?"
r6 = ask_docs(q6)
display_answer(q6, r6)

In [ ]:
q7 = "What webhook events can I subscribe to?"
r7 = ask_docs(q7)
display_answer(q7, r7)

## 21. Out-of-Scope Question

In [ ]:
q_oos = "Does CloudVault support SAML single sign-on?"
r_oos = ask_docs(q_oos)
display_answer(q_oos, r_oos)
print(f"\n  -> Expected LOW confidence. SAML/SSO not covered in our docs.")

## 22. Batch Q&A

In [ ]:
batch = [
    "How do I paginate through search results?",
    "How do I delete a file permanently?",
    "What happens when a webhook delivery fails?",
    "How do I verify a webhook signature?",
    "How long before deleted files are permanently removed?",
    "What error code means my token is expired or invalid?",
]

print("BATCH DOC QA")
print("=" * 80)
for q in batch:
    result = ask_docs(q)
    conf = result.get("confidence", "?")
    badge = {"high": "HIGH", "medium": "MED ", "low": "LOW "}.get(conf, "??? ")
    qtype = result.get("classified_type", "?")[:10]
    answer = result.get("answer", "")
    print(f"\n  [{badge}] [{qtype:>13}] Q: {q}")
    print(f"       A: {textwrap.shorten(answer, width=90, placeholder='...')}")

---

# Part E — Retrieval Evaluation

## 23. Why Evaluation Matters for Doc QA

In production doc QA systems, bad retrieval → wrong answer. You need to measure:

| Metric | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **Recall@k** | Is the correct section in the top-k results? | If retrieval misses, the LLM can't answer |
| **MRR** (Mean Reciprocal Rank) | How high does the correct section rank? | Higher rank = less noise for the LLM |
| **Section Hit Rate** | Does the top result come from the right page? | Wrong page = totally wrong context |

## 24. Ground-Truth Evaluation Set

We build a small evaluation set mapping questions to the doc section that should be retrieved.

In [ ]:
EVAL_SET = [
    {"question": "How do I get an API key?",
     "expected_page": "Authentication",
     "expected_section": "Getting an API Key"},
    {"question": "How do I refresh my OAuth token?",
     "expected_page": "Authentication",
     "expected_section": "Token Expiry and Refresh"},
    {"question": "What scopes can I give my API key?",
     "expected_page": "Authentication",
     "expected_section": "Scopes"},
    {"question": "How do I upload a file?",
     "expected_page": "File Operations",
     "expected_section": "Uploading a File"},
    {"question": "How do I download a file?",
     "expected_page": "File Operations",
     "expected_section": "Downloading a File"},
    {"question": "How do I list files in a folder?",
     "expected_page": "File Operations",
     "expected_section": "Listing Files in a Folder"},
    {"question": "How do I delete a file?",
     "expected_page": "File Operations",
     "expected_section": "Deleting a File"},
    {"question": "How do I search for files by content?",
     "expected_page": "Search",
     "expected_section": "Full-Text Search"},
    {"question": "How do I paginate search results?",
     "expected_page": "Search",
     "expected_section": "Pagination"},
    {"question": "How do I create a webhook?",
     "expected_page": "Webhooks",
     "expected_section": "Setting Up a Webhook"},
    {"question": "How do I verify a webhook signature?",
     "expected_page": "Webhooks",
     "expected_section": "Verifying Webhook Signatures"},
    {"question": "What are the API rate limits?",
     "expected_page": "Rate Limits & Errors",
     "expected_section": "Default Limits"},
    {"question": "How do I handle rate limiting in my code?",
     "expected_page": "Rate Limits & Errors",
     "expected_section": "Handling Rate Limits"},
    {"question": "What does a 401 error mean?",
     "expected_page": "Rate Limits & Errors",
     "expected_section": "Common Error Responses"},
]

print(f"Evaluation set: {len(EVAL_SET)} question-section pairs")
print(f"Pages covered: {len(set(e['expected_page'] for e in EVAL_SET))}")

## 25. Run Retrieval Evaluation

In [ ]:
def evaluate_retrieval(eval_set: list[dict], top_k: int = 5) -> dict:
    """Compute Recall@k, MRR, and page hit rate."""
    hits_at_k = 0
    reciprocal_ranks = []
    page_hits = 0
    details = []

    for item in eval_set:
        chunks = retrieve(item["question"], top_k=top_k)
        expected_page = item["expected_page"]
        expected_section = item["expected_section"]

        # Check page-level hit (top-1)
        top_page = chunks[0]["metadata"]["page"] if chunks else ""
        if top_page == expected_page:
            page_hits += 1

        # Check section-level hit and rank
        found_rank = None
        for rank, c in enumerate(chunks, 1):
            if (c["metadata"]["page"] == expected_page and
                expected_section in c["metadata"]["section"]):
                found_rank = rank
                break

        if found_rank is not None:
            hits_at_k += 1
            reciprocal_ranks.append(1.0 / found_rank)
        else:
            reciprocal_ranks.append(0.0)

        status = "HIT " if found_rank else "MISS"
        rank_str = f"@{found_rank}" if found_rank else "  -"
        details.append({
            "question": item["question"],
            "expected": f"{expected_page} > {expected_section}",
            "top_result": f"{chunks[0]['metadata']['page']} > {chunks[0]['metadata']['section']}" if chunks else "-",
            "status": status,
            "rank": found_rank,
        })

    n = len(eval_set)
    return {
        "recall_at_k": hits_at_k / n,
        "mrr": sum(reciprocal_ranks) / n,
        "page_hit_rate": page_hits / n,
        "n": n,
        "k": top_k,
        "details": details,
    }


eval_results = evaluate_retrieval(EVAL_SET, top_k=5)

print("RETRIEVAL EVALUATION")
print("=" * 80)
print(f"  Recall@{eval_results['k']}: {eval_results['recall_at_k']:.1%}")
print(f"  MRR:       {eval_results['mrr']:.3f}")
print(f"  Page top-1 hit rate: {eval_results['page_hit_rate']:.1%}")
print(f"  Questions: {eval_results['n']}")

print(f"\nDetails:")
for d in eval_results["details"]:
    rank_str = f"@{d['rank']}" if d["rank"] else " - "
    print(f"  {d['status']} {rank_str:>3} | Q: {d['question'][:45]}")
    if d["status"] == "MISS":
        print(f"            Expected: {d['expected']}")
        print(f"            Got:      {d['top_result']}")

## 26. Evaluation at Different k Values

In [ ]:
print("RECALL AND MRR AT DIFFERENT k")
print("=" * 50)
print(f"{'k':>3} | {'Recall@k':>10} | {'MRR':>8} | {'Page Hit':>10}")
print("-" * 50)
for k in [1, 3, 5, 10]:
    r = evaluate_retrieval(EVAL_SET, top_k=k)
    print(f"  {k:>1} | {r['recall_at_k']:>9.1%} | {r['mrr']:>7.3f} | {r['page_hit_rate']:>9.1%}")

print(f"\nInterpretation:")
print(f"  Recall@1 = section hit rate at top-1 (strictest)")
print(f"  Recall@5 = does the right section appear anywhere in top-5?")
print(f"  MRR = average of 1/rank (higher = better ranking)")

## 27. Answer Quality Evaluation (LLM-as-Judge)

Beyond retrieval, we also want to know if the generated answer is correct, grounded, and well-cited.

In [ ]:
JUDGE_SYSTEM = "You evaluate documentation assistant answers for quality."

JUDGE_PROMPT = """Evaluate this documentation QA answer.

QUESTION: {question}
ANSWER: {answer}
SOURCES: {sources}

Score each dimension 1-5:
- correctness: is the answer factually right based on the sources?
- grounding: does every claim have a matching source passage?
- completeness: does the answer cover all parts of the question?
- citation_quality: are citations properly formatted and accurate?
- usefulness: would a developer find this answer helpful?

Return ONLY JSON:
{{"correctness": N, "grounding": N, "completeness": N,
  "citation_quality": N, "usefulness": N,
  "overall": N, "notes": "brief explanation"}}"""

# Judge a sample of answers
judge_samples = [
    ("How do I upload a file using the API?", r1),
    ("What rate limits apply on the standard plan?", r5),
    ("I'm getting a 403 forbidden error. What's wrong?", r6),
]

print("ANSWER QUALITY EVALUATION (LLM-as-Judge)")
print("=" * 80)

for q, result in judge_samples:
    sources_text = ", ".join(result.get("sources_used", [])[:3])
    resp = ask(
        JUDGE_PROMPT.format(
            question=q,
            answer=result.get("answer", "")[:600],
            sources=sources_text,
        ),
        system=JUDGE_SYSTEM,
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(resp)
    if scores:
        print(f"\n  Q: {q[:60]}")
        for dim in ["correctness", "grounding", "completeness", "citation_quality", "usefulness"]:
            val = scores.get(dim, "?")
            bar = "*" * int(val) if isinstance(val, (int, float)) else "?"
            print(f"    {dim:20s}: {val}/5 {bar}")
        print(f"    {'overall':20s}: {scores.get('overall', '?')}/5")
        if scores.get("notes"):
            print(f"    Notes: {scores['notes'][:80]}")

## 28. More Evaluation Ideas

### Ideas you would use in a production doc QA system

| Evaluation Strategy | What It Measures | When to Use |
|--------------------|-----------------|-------------|
| **Ground-truth QA pairs** | End-to-end answer correctness | Before deploying a new model or prompt |
| **Retrieval A/B test** | Compare two chunk sizes or embedding models | When experimenting with retrieval params |
| **Citation verification** | Does each `[Doc: ...]` citation actually appear in the source? | Continuous monitoring |
| **Out-of-scope detection rate** | How often does the system correctly say "not in the docs" | Before launch |
| **User satisfaction** | Thumbs up/down on each answer | In production |
| **Latency tracking** | Time from question to answer | Always |
| **Embedding drift** | When docs update, do old embeddings still find the right sections? | After doc updates |

### Building an Evaluation Pipeline

```
1. Write 30-50 question-section pairs covering all doc pages
2. Compute Recall@k and MRR as a baseline
3. Change one variable (chunk size, embedding model, top-k)
4. Recompute metrics and compare
5. Add LLM-as-judge for answer quality in addition to retrieval metrics
6. Track metrics over time as documentation evolves
```

---

## 29. Common Mistakes

| Mistake | Why It's Bad | Better Approach |
|---------|-------------|----------------|
| Splitting code blocks across chunks | Produces two broken snippets the LLM can't use | Keep heading + prose + code as one unit |
| Using tiny chunks (200 chars) for docs | Most doc sections need context around the code | Use 600-700 char chunks for documentation |
| No metadata on chunks | Can't filter by page or section | Always tag page, section, has_code |
| No evaluation dataset | No way to measure if retrieval is correct | Build ground-truth question-section pairs |
| Treating all questions the same | How-to needs steps, reference needs a value, troubleshooting needs cause/fix | Classify question type first |
| Hallucinating endpoints | LLM may invent API paths not in the docs | System prompt forbids inventing |

## 30. Production Improvement Ideas

1. **Parse real docs** — ingest markdown/RST/HTML from your actual documentation site
2. **Hybrid search** — combine BM25 keyword search with vector search for better retrieval of parameter names
3. **Versioned docs** — tag chunks with doc version so answers reference the right API release
4. **Conversation memory** — maintain multi-turn context so users can ask follow-ups
5. **Code execution** — validate generated code examples by running them in a sandbox
6. **Feedback loop** — collect thumbs up/down and use to improve retrieval or prompts
7. **Auto-index on commit** — re-index documentation whenever the docs repo is updated

## 31. Exercises

### Exercise 1
Add a sixth doc page about **Folder Operations** (create, rename, delete folders). Write 3 evaluation pairs for it and re-run the retrieval evaluation to see if recall changes.

### Exercise 2
Implement a `code_only_retrieval(question)` function that retrieves only chunks tagged with `has_code=true`, then generates answers using only those code-heavy chunks. Compare against the default retriever.

### Exercise 3
Build a citation verifier: after generating an answer, check each `[Doc: page > section]` citation against the actual retrieved chunks. Report how many citations are valid vs hallucinated.

### Mini Challenge
Build a multi-turn doc chat: the user asks a first question, gets an answer, then asks a follow-up. The system should use the previous answer as additional context while still citing the official docs.

In [ ]:
print("SESSION SUMMARY")
print("=" * 60)
print(f"Documentation pages indexed: {len(DOCS)}")
print(f"Sections parsed: {len(all_sections)}")
print(f"Chunks indexed: {len(all_chunks)}")
print(f"Evaluation pairs: {len(EVAL_SET)}")
print(f"Retrieval Recall@5: {eval_results['recall_at_k']:.1%}")
print(f"Retrieval MRR: {eval_results['mrr']:.3f}")
print(f"\nCapabilities built:")
print(f"  - Doc parsing with section + code metadata")
print(f"  - Question type classification")
print(f"  - Type-aware answer generation with citations")
print(f"  - Retrieval evaluation (Recall@k, MRR, page hit rate)")
print(f"  - LLM-as-judge answer quality scoring")

## 32. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Keep code blocks intact** — never split a code example across chunk boundaries |
| 2 | **Tag chunks with metadata** — page, section, has_code, has_table |
| 3 | **Classify question type first** — how-to, reference, troubleshooting, conceptual, listing |
| 4 | **Match answer format to question type** — steps for how-to, exact values for reference |
| 5 | **Cite docs explicitly** — every claim needs a `[Doc: page > section]` reference |
| 6 | **Build an evaluation set** — question-section pairs for measuring Recall@k and MRR |
| 7 | **Evaluate at multiple k values** — Recall@1 is strict, Recall@5 is more forgiving |
| 8 | **Use LLM-as-judge** for answer quality in addition to retrieval metrics |
| 9 | **Forbid invention** — the system prompt must prevent the LLM from making up endpoints |
| 10 | **Track metrics over time** — doc updates can break previously correct retrieval |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*